# MAPED: synthetic data on CPU

Generate a small tilt series, align it step by step, and inspect the merged patterns.

## Setup

**Development prerequisite:** this workflow still requires MAPED development builds; the package command alone does not yet provide these APIs.

New to Python? Follow the [Conda or uv setup guide](https://github.com/bobleesj/quantem-tutorials/blob/oct-17-maped-tutorials/tutorials/diffraction/maped/setup/README.md). If packages are missing in an existing Python 3.11+ environment, uncomment the cell below, then restart the kernel.

In [ ]:
# %pip install --upgrade quantem pandas

In [ ]:
from pathlib import Path

from quantem.core.datastructures import Dataset4dstem
from quantem.diffraction import MAPED

folder = Path("synthetic-tilts")

## Generate data

Five 64 × 64 scans, each with 64 × 64 detector pixels, known shifts, and Poisson count noise. The generator is included here: no companion script or download is needed.

This is a registration example, not a microscope simulation. The illustrative scan step is 1 Å. The generator records the shifts in `ground_truth.json`.

In [ ]:
import json

import h5py
import hdf5plugin
import torch

SCAN_SHIFTS = [[0, 0], [2.3, -1.4], [-1.7, 3.1], [3.4, 2.2], [-2.6, -3.3]]
DETECTOR_SHIFTS = [[0, 0], [-1.2, 5.5], [2.1, -11.8], [-3, 5.1], [3.6, 2.9]]


def gaussian(row_t, col_t, center_row, center_col, sigma, amplitude=1):
    """Evaluate an analytic Gaussian at the supplied coordinates."""
    return amplitude * torch.exp(
        -((row_t - center_row) ** 2 + (col_t - center_col) ** 2) / (2 * sigma**2)
    )


def generate(folder: Path) -> None:
    """Write five 64 x 64 scans with 64 x 64 diffraction patterns and known shifts."""
    folder.mkdir(parents=True, exist_ok=True)
    generator = torch.Generator(device="cpu").manual_seed(0)
    row_t = torch.arange(64, dtype=torch.float32)[:, None]
    col_t = torch.arange(64, dtype=torch.float32)[None, :]
    for index, (scan, detector) in enumerate(zip(SCAN_SHIFTS, DETECTOR_SHIFTS, strict=True)):
        scan_row_t, scan_col_t = row_t - scan[0], col_t - scan[1]
        image_t = (
            3
            - gaussian(scan_row_t, scan_col_t, 24, 20, 15)
            + gaussian(scan_row_t, scan_col_t, 40, 44, 5, 0.7)
            + gaussian(scan_row_t, scan_col_t, 30, 38, 3, 0.5)
        )
        detector_row_t, detector_col_t = row_t - detector[0], col_t - detector[1]
        pattern_t = (
            gaussian(detector_row_t, detector_col_t, 32, 32, 2, 3)
            + gaussian(detector_row_t, detector_col_t, 18, 42, 2, 0.35)
            + gaussian(detector_row_t, detector_col_t, 42, 46, 2, 0.25)
            + gaussian(detector_row_t, detector_col_t, 46, 22, 2, 0.25)
            + gaussian(detector_row_t, detector_col_t, 22, 18, 2, 0.25)
        )
        expected_t = 200 * image_t[:, :, None, None] * pattern_t
        counts_t = torch.poisson(expected_t, generator=generator).to(torch.uint16)
        with h5py.File(folder / f"tilt_{index}_master.h5", "w") as file:
            file.attrs["scan_shape"] = (64, 64)
            file.attrs["name"] = f"Synthetic tilt {index}"
            file.attrs["synthetic"] = True
            file.create_dataset(
                "entry/data/data_000001",
                data=counts_t.numpy().reshape(-1, 64, 64),
                chunks=(1, 64, 64),
                **hdf5plugin.Bitshuffle(cname="lz4"),
            )
        with h5py.File(folder / f"tilt_{index}.h5", "w") as file:
            file.attrs["synthetic"] = True
            metadata = file.create_group("STEM Metadata")
            metadata.attrs["Pixel Size"] = [0.1]  # Synthetic nm/pixel, not measured.
            metadata.attrs["NumPixelsX"] = [64]
            metadata.attrs["NumPixelsY"] = [64]
            metadata.attrs["IsPixelSizeCalibrated"] = [1]
    truth = {
        "source": "https://github.com/electronmicroscopy/quantem/pull/256",
        "source_notebook": "https://github.com/user-attachments/files/29561645/maped_torch_validation.ipynb",
        "description": "Analytic Gaussians and shifts from PR256, evaluated at displaced coordinates; Poisson integer counts replace additive Gaussian noise.",
        "seed": 0,
        "shape": [64, 64, 64, 64],
        "tilts": 5,
        "scan_shifts_px": SCAN_SHIFTS,
        "detector_shifts_px": DETECTOR_SHIFTS,
        "shift_tolerance_px": 0.2,
        "minimum_bright_field_ncc": 0.95,
        "units": ["pixels", "pixels", "pixels", "pixels"],
        "synthetic_scan_step_A": 1.0,
    }
    (folder / "ground_truth.json").write_text(json.dumps(truth, indent=2) + "\n")


generate(folder)

## Load data

Load the generated master files. The synthetic counts need no detector correction.

In [ ]:
files = sorted(folder.glob("*_master.h5"))
tilts = [Dataset4dstem.from_file(file, "arina") for file in files]
maped = MAPED.from_datasets(tilts)
maped.show_tilts(num_cols=5, size="medium")

## Align

`align_diffraction()` measures detector shifts; `align_images()` measures scan shifts from the default whole-detector mean images. Shifts use `(row, col)` pixels.

To use BF, ADF, DF or simultaneously recorded HAADF images, pass `images=images` to `align_images()`: one 2D array per tilt, with the same scan shape, sampling and orientation. The merge still uses the original diffraction data.

In [ ]:
maped.find_beam_center(show=False)
maped.align_diffraction(show=False)
maped.align_images(iterations=80, show=False)

In [ ]:
maped.show_alignment(num_cols=5, size="medium", reference=0)  # Set the zero-tilt index.

## Merge

Keep the common field of view and preserve the computed float32 values.

In [ ]:
merged = maped.merge(crop=True, show=False)
maped.report()  # Table; use as_dict=True for a dictionary.

## Inspect results

Compare individual diffraction patterns before and after merging at three marked scan positions.

In [ ]:
maped.show(positions=[(15, 15), (25, 25), (40, 40)], tilt_indices=[0])

## Save and reopen (optional)

Native QuantEM ZIP serialization preserves the merged values and alignment metadata.

In [ ]:
# from quantem.core.io import load
# saved = Path("outputs/merged_synthetic_cpu.zip")
# saved.parent.mkdir(parents=True, exist_ok=True)
# merged.save(saved)
# reopened = load(saved)
# reopened

Built on Henry Bell’s [QuantEM PR #256](https://github.com/electronmicroscopy/quantem/pull/256). Synthetic objects and shifts follow its [validation notebook](https://github.com/user-attachments/files/29561645/maped_torch_validation.ipynb).